# Diving into Portal Administration API

- we saw how we can query and work with the Portal logs, but we can go deeper

In [ ]:
from arcgis.gis import GIS
gis = GIS(profile='your_enterprise_profile')

In [ ]:
admin = gis.admin

#### The `admin` endpoint

- This endpoint allows us to modify the UX to update the password policies.
- Provides access into the back system of the Enterprise site


##### Underlying Portal Machine

In [ ]:
machines = admin.machines
machines.properties

In [ ]:
machine = machines.list()[0]

In [ ]:
machine.status()

### System Operations

- there are times when you add lots of content, the indexes can come out of sync.

In [ ]:
system = admin.system
system

In [ ]:
system.index_status

In [ ]:
r = system.reindex(mode="SEARCH_MODE")

In [ ]:
system.index_status

### Looking at Registered Server

In [ ]:
servers = gis.admin.servers
servers

In [ ]:
server = servers.list()[0]
server

##### Examine Server Usage

In [ ]:
mgr = server.usage
mgr

In [ ]:
report = mgr.list()[-1]
report

#### Who is Viewing Our Data?

In [ ]:
import pandas as pd
df = pd.DataFrame(report.query()['report']['report-data'][0][0]['data'][1])
series = df.groupby("user")['count'].sum()
series

In [ ]:
series.plot(kind='bar')

### Managing Services and Folders on the Hosting Server

- like the `enterprise` we can create folders

In [ ]:
services = server.services

In [ ]:
services.folders

##### Managing Folders

In [ ]:
services.create_folder("devsummit2026")

In [ ]:
services.folders

In [ ]:
services.delete_folder("devsummit2026")

In [ ]:
services.folders

#### Demo: Listing all Services

In [ ]:
for folder in services.folders:
    for s in services.list(folder):
        print(s)

#### Control a Service's State

- `start`, `stop` and `restart` services

In [ ]:
for service in services.list():
    if service.properties.serviceName == 'SampleWorldCities':
        break
service

**Check the Service Status**

- Shows if the services is running or not

In [ ]:
service.status

In [ ]:
service.stop()

In [ ]:
service.status

In [ ]:
service.start()

#### Modifying a Service

- modify extensions, pooling, etc..

In [ ]:
service

**Enable KML on the Service**

In [ ]:
for ext in service.extensions:
    if ext.typeName == "KmlServer":
        ext.enabled = True
[(ext.typeName, ext.enabled) for ext in service.extensions]

### Server Logs

- ArcGIS Server records events that occur, and any errors associated with those events, to logs. Logs are an important tool for monitoring and troubleshooting problems with your site. Information in the logs will help you identify errors and provide context on how to address problems

In [ ]:
logs = server.logs
logs

In [ ]:
import datetime
import pandas as pd
now = datetime.datetime.now()
start_time = now - datetime.timedelta(days=10)
start_time

In [ ]:
recent_logs = logs.query(start_time = start_time)

#print a message as a sample
pd.DataFrame(recent_logs['logMessages']).head()